# Patrones frecuentes de compra con restricciones

Material docente · Datos ficticios · Python / Jupyter / Google Colab

**Objetivo:** resolver los seis apartados del enunciado y verificar la equivalencia entre FP-Growth y CP-SAT. El notebook es autocontenido: no descarga datasets ni necesita archivos auxiliares.

Ejecute las celdas en orden. La instalación solo se realiza si faltan paquetes. Se usa `mlxtend` como implementación de FP-Growth en lugar de `pyfim`, manteniendo el objetivo del material original. No se pretende implementar FP-Growth desde cero.

In [ ]:
import importlib.util
import subprocess
import sys
paquetes = {"pandas": "pandas", "mlxtend": "mlxtend", "ortools": "ortools"}
faltantes = [pkg for modulo, pkg in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from ortools.sat.python import cp_model
from itertools import combinations
from fractions import Fraction
from math import ceil
from importlib.metadata import version
from IPython.display import display
print({p: version(p) for p in paquetes.values()})

{'pandas': '3.1.0', 'mlxtend': '0.25.0', 'ortools': '9.15.6755'}


## 1. Preparación y cálculo manual
Cada fila representa una compra independiente. Las repeticiones de compras se conservan, pues corresponden a transacciones distintas. Dentro de una compra solo importa la presencia de cada producto.

In [ ]:
transacciones = [['pan', 'leche', 'huevos'], ['pan', 'leche'], ['pan', 'huevos'], ['leche', 'huevos'], ['pan', 'leche', 'mantequilla'], ['pan', 'mantequilla'], ['pan', 'leche', 'huevos', 'mantequilla'], ['arroz', 'atun'], ['arroz', 'atun', 'leche'], ['arroz', 'pan'], ['pan', 'leche', 'cafe'], ['leche', 'cafe'], ['pan', 'cafe'], ['pan', 'leche', 'huevos'], ['arroz', 'atun', 'pan'], ['pan', 'leche', 'mantequilla'], ['leche', 'huevos', 'cafe'], ['pan', 'leche', 'cafe', 'mantequilla'], ['arroz', 'atun', 'cafe'], ['pan', 'huevos', 'mantequilla']]
transacciones = [sorted(set(t)) for t in transacciones]
N = len(transacciones)
encoder = TransactionEncoder()
D = pd.DataFrame(encoder.fit(transacciones).transform(transacciones), columns=encoder.columns_)
productos = list(D.columns)
display(pd.DataFrame({"compra": range(1, N+1), "productos": transacciones}))
display(D.astype(int))
assert D.shape == (20, 7)

def frecuencia(itemset):
    return sum(set(itemset).issubset(t) for t in transacciones)

def umbral_absoluto(soporte):
    # El soporte mínimo se convierte a una cantidad entera con techo.
    return ceil(Fraction(str(soporte)) * N)

soporte_minimo = 0.20
min_freq = umbral_absoluto(soporte_minimo)
display(pd.DataFrame([{"conjunto": str(sorted(x)), "frecuencia": frecuencia(x),
                      "soporte": frecuencia(x)/N}
                     for x in [{"pan"}, {"leche"}, {"pan", "leche"}]]))
print("Frecuencia mínima:", min_freq)

Frecuencia mínima: 4


,compra,productos
0,1,"[huevos, leche, pan]"
1,2,"[leche, pan]"
2,3,"[huevos, pan]"
3,4,"[huevos, leche]"
4,5,"[leche, mantequilla, pan]"
5,6,"[mantequilla, pan]"
6,7,"[huevos, leche, mantequilla, pan]"
7,8,"[arroz, atun]"
8,9,"[arroz, atun, leche]"
9,10,"[arroz, pan]"


,arroz,atun,cafe,huevos,leche,mantequilla,pan
0,0,0,0,1,1,0,1
1,0,0,0,0,1,0,1
2,0,0,0,1,0,0,1
3,0,0,0,1,1,0,0
4,0,0,0,0,1,1,1
5,0,0,0,0,0,1,1
6,0,0,0,1,1,1,1
7,1,1,0,0,0,0,0
8,1,1,0,0,1,0,0
9,1,0,0,0,0,0,1


,conjunto,frecuencia,soporte
0,['pan'],14,0.7
1,['leche'],12,0.6
2,"['leche', 'pan']",8,0.4


**Respuesta manual:** pan aparece en 14 compras (70 %), leche en 12 (60 %) y ambos en 8 (40 %). Un soporte del 20 % exige al menos 4 de las 20 compras.

Para un conjunto $X$, $f(X)=|\{t:X\subseteq t\}|$ y $supp(X)=f(X)/N$. El soporte mide presencia conjunta, sin dirección.

## 2. FP-Growth
La entrada es una matriz booleana. El umbral de `mlxtend` es una proporción entre 0 y 1; no un porcentaje entero. No se limita el tamaño de los patrones en esta primera búsqueda.

In [ ]:
def buscar_fp(soporte):
    resultado = fpgrowth(D, min_support=soporte, use_colnames=True)
    # Se conservan frozenset para comparar sin depender del orden de los productos.
    resultado["frecuencia"] = resultado["itemsets"].map(frecuencia)
    resultado["tamano"] = resultado["itemsets"].map(len)
    return resultado

def presentar(diccionario):
    filas = [{"productos": ", ".join(sorted(x)), "tamano": len(x),
              "frecuencia": f, "soporte": f/N} for x, f in diccionario.items()]
    return pd.DataFrame(filas, columns=["productos", "tamano", "frecuencia", "soporte"]).sort_values(
        ["frecuencia", "productos"], ascending=[False, True]).reset_index(drop=True)

fp = buscar_fp(soporte_minimo)
mapa_fp = dict(zip(fp.itemsets, fp.frecuencia))
display(presentar(mapa_fp))
display(presentar({x:f for x,f in mapa_fp.items() if len(x)>=2}))

,productos,tamano,frecuencia,soporte
0,pan,1,14,0.70
1,leche,1,12,0.60
2,"leche, pan",2,8,0.40
3,huevos,1,7,0.35
4,cafe,1,6,0.30
5,mantequilla,1,6,0.30
6,"mantequilla, pan",2,6,0.30
7,arroz,1,5,0.25
8,"huevos, leche",2,5,0.25
9,"huevos, pan",2,5,0.25


,productos,tamano,frecuencia,soporte
0,"leche, pan",2,8,0.40
1,"mantequilla, pan",2,6,0.30
2,"huevos, leche",2,5,0.25
3,"huevos, pan",2,5,0.25
4,"arroz, atun",2,4,0.20
5,"cafe, leche",2,4,0.20
6,"leche, mantequilla",2,4,0.20
7,"leche, mantequilla, pan",3,4,0.20


## 3. Formulación con OR-Tools
Sea $D_{ti}=1$ si la transacción $t$ contiene el producto $i$.

- $I_i\in\{0,1\}$: seleccionamos el producto $i$.
- $T_t\in\{0,1\}$: la transacción $t$ contiene todo el conjunto seleccionado.
- $f=\sum_t T_t$: frecuencia absoluta del conjunto.

Definimos la cantidad de productos seleccionados ausentes en una compra:

$$a_t=\sum_i I_i(1-D_{ti}).$$

La cobertura exacta exige $T_t=1\iff a_t=0$. Se implementa mediante dos implicaciones:

$$T_t=1\Rightarrow a_t=0,\qquad T_t=0\Rightarrow a_t\geq1.$$

La segunda impide declarar como no cubierta una compra que sí contiene el conjunto. Con solo la primera, el solver podría subestimar la frecuencia.

Finalmente, $\sum_i I_i\geq1$ y $f\geq\lceil N\cdot minsup\rceil$. **No hay función objetivo:** buscamos todas las soluciones factibles, no solo un patrón óptimo.

In [ ]:
class Recolector(cp_model.CpSolverSolutionCallback):
    def __init__(self, I, f, productos):
        super().__init__()
        self.I, self.f, self.productos = I, f, productos
        self.resultados = {}

    def on_solution_callback(self):
        conjunto = frozenset(p for p, var in zip(self.productos, self.I) if self.value(var))
        self.resultados[conjunto] = self.value(self.f)


def buscar_cp(soporte, comercial=False):
    modelo = cp_model.CpModel()
    I = [modelo.new_bool_var(f"I_{p}") for p in productos]
    T = [modelo.new_bool_var(f"T_{t}") for t in range(N)]
    f = modelo.new_int_var(0, N, "frecuencia")
    for t in range(N):
        ausentes = sum(I[i] * (1-int(D.iloc[t, i])) for i in range(len(productos)))
        modelo.add(ausentes == 0).only_enforce_if(T[t])
        modelo.add(ausentes >= 1).only_enforce_if(T[t].Not())
    modelo.add(f == sum(T))
    modelo.add(f >= umbral_absoluto(soporte))
    modelo.add(sum(I) >= 1)

    if comercial:
        indice = {p:i for i,p in enumerate(productos)}
        modelo.add(sum(I) >= 2)
        modelo.add(sum(I) <= 3)
        modelo.add(I[indice["leche"]] == 1)
        modelo.add(I[indice["cafe"]] + I[indice["mantequilla"]] <= 1)

    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.num_search_workers = 1
    recolector = Recolector(I, f, productos)
    estado = solver.solve(modelo, recolector)
    # Sin límite temporal para este pequeño dataset. No aceptar una enumeración parcial.
    if estado not in (cp_model.OPTIMAL, cp_model.INFEASIBLE):
        raise RuntimeError(f"Enumeración incompleta: {solver.status_name(estado)}")
    return recolector.resultados, solver.status_name(estado)

mapa_cp, estado = buscar_cp(soporte_minimo)
print("Estado:", estado, "| Patrones:", len(mapa_cp))
display(presentar(mapa_cp))
assert mapa_cp == mapa_fp, "Diferencia entre patrones o frecuencias"
print("Verificado: FP-Growth y CP-SAT coinciden en conjuntos y frecuencias.")

Estado: OPTIMAL | Patrones: 15
Verificado: FP-Growth y CP-SAT coinciden en conjuntos y frecuencias.


,productos,tamano,frecuencia,soporte
0,pan,1,14,0.70
1,leche,1,12,0.60
2,"leche, pan",2,8,0.40
3,huevos,1,7,0.35
4,cafe,1,6,0.30
5,mantequilla,1,6,0.30
6,"mantequilla, pan",2,6,0.30
7,arroz,1,5,0.25
8,"huevos, leche",2,5,0.25
9,"huevos, pan",2,5,0.25


## 4. Condiciones comerciales
En la función anterior, el bloque `comercial` incorpora tres condiciones: tamaño de 2 a 3, inclusión de leche e incompatibilidad de cafe con mantequilla. La incompatibilidad es una decisión hipotética del negocio; no se deduce de los datos.

En FP-Growth obtenemos los patrones y luego los filtramos. En CP-SAT las condiciones forman parte del modelo y pueden descartar opciones durante la búsqueda. Esto ofrece flexibilidad, pero no implica que CP-SAT sea siempre más rápido.

In [ ]:
def cumple_condiciones(x):
    return 2 <= len(x) <= 3 and "leche" in x and not {"cafe", "mantequilla"}.issubset(x)

fp_comercial = {x:f for x,f in mapa_fp.items() if cumple_condiciones(x)}
cp_comercial, estado = buscar_cp(soporte_minimo, comercial=True)
assert cp_comercial == fp_comercial
print("Estado:", estado, "| Patrones comerciales:", len(cp_comercial))
display(presentar(cp_comercial))

Estado: OPTIMAL | Patrones comerciales: 5


,productos,tamano,frecuencia,soporte
0,"leche, pan",2,8,0.40
1,"huevos, leche",2,5,0.25
2,"cafe, leche",2,4,0.20
3,"leche, mantequilla",2,4,0.20
4,"leche, mantequilla, pan",3,4,0.20


**Interpretación:** son candidatos por coocurrencia y condiciones comerciales. La tabla permite identificar compras conjuntas, pero no contiene costos, márgenes ni respuesta a descuentos; por ello no permite asegurar rentabilidad.

## 5. Reglas de asociación
Para cada conjunto frecuente $X$ con al menos dos productos, recorremos sus subconjuntos propios no vacíos $A$ y definimos $B=X\setminus A$.

$$supp(A\to B)=\frac{f(A\cup B)}{N},\quad conf(A\to B)=\frac{f(A\cup B)}{f(A)},\quad lift(A\to B)=\frac{conf(A\to B)}{f(B)/N}.$$

Se calculan explícitamente las métricas para observar su significado. Se usan los patrones originales, sin condiciones comerciales. Conservar solo patrones comerciales eliminaría subconjuntos necesarios para formar reglas.

In [ ]:
reglas = []
for X, fX in mapa_fp.items():
    for tam in range(1, len(X)):
        for elementos in combinations(sorted(X), tam):
            A = frozenset(elementos)
            B = X - A
            confianza = fX / frecuencia(A)
            if confianza >= 0.70:
                reglas.append({"antecedente": ", ".join(sorted(A)),
                               "consecuente": ", ".join(sorted(B)),
                               "soporte": fX/N, "confianza": confianza,
                               "lift": confianza/(frecuencia(B)/N)})
reglas_df = pd.DataFrame(reglas).sort_values(
    ["confianza", "lift", "antecedente", "consecuente"],
    ascending=[False, False, True, True]).reset_index(drop=True)
display(reglas_df.round(4))
print("Número de reglas:", len(reglas_df))
# Ejemplo independiente para interpretar una regla concreta.
A, B = {"mantequilla"}, {"pan"}
fAB = frecuencia(A | B)
print("mantequilla -> pan:", {"soporte": fAB/N,
      "confianza": fAB/frecuencia(A),
      "lift": (fAB/frecuencia(A))/(frecuencia(B)/N)})

Número de reglas: 6
mantequilla -> pan: {'soporte': 0.3, 'confianza': 1.0, 'lift': 1.4285714285714286}


,antecedente,consecuente,soporte,confianza,lift
0,atun,arroz,0.20,1.0000,4.0000
1,"leche, mantequilla",pan,0.20,1.0000,1.4286
2,mantequilla,pan,0.30,1.0000,1.4286
3,arroz,atun,0.20,0.8000,4.0000
4,huevos,leche,0.25,0.7143,1.1905
5,huevos,pan,0.25,0.7143,1.0204


**Interpretación de mantequilla → pan:** las 6 compras con mantequilla incluyen pan. El soporte de la regla es 6/20 = 30 %, su confianza es 100 % y su lift es 1/0.70 ≈ 1.43. En esta muestra, la proporción de compras con pan entre las que contienen mantequilla supera su proporción general. No implica que comprar mantequilla cause comprar pan.

Una confianza alta puede reflejar que el consecuente ya es muy común. El lift aporta esa referencia: 1 indica la proporción esperada bajo independencia, mayor que 1 una asociación positiva y menor que 1 una negativa. Estos valores describen una muestra pequeña y ficticia; no prueban significancia estadística ni rentabilidad.

## 6. Sensibilidad y validación
Se repite la comparación para cuatro umbrales. Además, por haber solo siete productos, comprobamos los resultados contra una enumeración exhaustiva independiente de los 127 conjuntos no vacíos. Esta última se usa como verificación didáctica, no como alternativa escalable.

In [ ]:
def fuerza_bruta(soporte, comercial=False):
    resultados = {}
    for tam in range(1, len(productos)+1):
        for valores in combinations(productos, tam):
            x = frozenset(valores)
            f = frecuencia(x)
            if f >= umbral_absoluto(soporte) and (not comercial or cumple_condiciones(x)):
                resultados[x] = f
    return resultados

resumen = []
for soporte in [0.10, 0.20, 0.30, 0.40]:
    tabla = buscar_fp(soporte)
    base = dict(zip(tabla.itemsets, tabla.frecuencia))
    for comercial in [False, True]:
        esperado = {x:f for x,f in base.items() if not comercial or cumple_condiciones(x)}
        obtenido, estado = buscar_cp(soporte, comercial)
        assert esperado == obtenido == fuerza_bruta(soporte, comercial)
        resumen.append({"soporte_minimo": soporte,
                        "frecuencia_minima": umbral_absoluto(soporte),
                        "condiciones_comerciales": comercial,
                        "patrones_FP": len(esperado), "patrones_CP": len(obtenido),
                        "coinciden": True})
sensibilidad = pd.DataFrame(resumen)
display(sensibilidad)
assert frecuencia({"pan"}) == 14
assert frecuencia({"leche"}) == 12
assert frecuencia({"pan", "leche"}) == 8
print("Todas las comprobaciones se completaron correctamente.")

Todas las comprobaciones se completaron correctamente.


,soporte_minimo,frecuencia_minima,condiciones_comerciales,patrones_FP,patrones_CP,coinciden
0,0.1,2,False,21,21,True
1,0.1,2,True,7,7,True
2,0.2,4,False,15,15,True
3,0.2,4,True,5,5,True
4,0.3,6,False,7,7,True
5,0.3,6,True,1,1,True
6,0.4,8,False,3,3,True
7,0.4,8,True,1,1,True


**Conclusiones y respuestas:**

1. Aumentar el soporte mínimo mantiene o reduce el número de patrones: ningún conjunto que incumpla un umbral menor puede satisfacer uno mayor. En estas pruebas las tablas permiten observar la reducción concreta.
2. Las condiciones comerciales mantienen o reducen el conjunto de soluciones para un mismo soporte. Si no quedaran patrones, se trataría de un resultado válido, no necesariamente de un error del programa.
3. FP-Growth es un algoritmo especializado en patrones frecuentes. CP-SAT permite expresar condiciones adicionales en un modelo declarativo. La coincidencia de resultados verifica la formulación para este dataset y los escenarios probados.
4. El modelo debe representar una equivalencia de cobertura. La frecuencia debe contar **todas** las transacciones que contienen el patrón.
5. La enumeración puede crecer exponencialmente con el número de productos. Este ejercicio pequeño facilita entender y verificar el modelo; no demuestra rendimiento en datos masivos.
6. Una evaluación de promociones requeriría datos reales, costos, márgenes y validación de su efecto.

## Referencias
- Material del curso proporcionado: `CS-Topics-Week10-1 (1).pdf` y `22_1_TopicosCC_10 (2).ipynb`. Se retoman los conceptos de soporte, cobertura y variables por ítem y transacción.
- [Documentación oficial de FP-Growth en mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/fpgrowth/).
- [Documentación oficial del solver CP-SAT](https://developers.google.com/optimization/cp/cp_solver).

Las versiones utilizadas quedan registradas en la primera salida. Los datos se fijan explícitamente; el orden interno de enumeración del solver no afecta las tablas ordenadas ni las comprobaciones.